# 🔍 Module 3.3: Search & Filter Runs

**Goal**: Master `mlflow.search_runs()` to query, filter, sort, and analyze experiment results programmatically.

---

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.autolog(disable=True)
mlflow.set_experiment("03_search_and_filter")

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Step 1: Generate Multiple Runs

Let's create a rich set of runs to search through.

In [ ]:
# Generate diverse runs for searching
run_configs = [
    ("dt_shallow", DecisionTreeClassifier(max_depth=3, random_state=42),
     {"model_type": "DecisionTree", "max_depth": 3}, {"purpose": "baseline", "model_family": "tree"}),
    ("dt_deep", DecisionTreeClassifier(max_depth=10, random_state=42),
     {"model_type": "DecisionTree", "max_depth": 10}, {"purpose": "experiment", "model_family": "tree"}),
    ("rf_small", RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
     {"model_type": "RandomForest", "n_estimators": 50, "max_depth": 5}, {"purpose": "baseline", "model_family": "ensemble"}),
    ("rf_medium", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
     {"model_type": "RandomForest", "n_estimators": 100, "max_depth": 8}, {"purpose": "experiment", "model_family": "ensemble"}),
    ("rf_large", RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42),
     {"model_type": "RandomForest", "n_estimators": 300, "max_depth": 15}, {"purpose": "experiment", "model_family": "ensemble"}),
    ("gb_default", GradientBoostingClassifier(n_estimators=100, random_state=42),
     {"model_type": "GradientBoosting", "n_estimators": 100}, {"purpose": "baseline", "model_family": "ensemble"}),
    ("gb_tuned", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
     {"model_type": "GradientBoosting", "n_estimators": 200, "learning_rate": 0.05}, {"purpose": "experiment", "model_family": "ensemble"}),
    ("lr_default", LogisticRegression(max_iter=1000, random_state=42),
     {"model_type": "LogisticRegression", "max_iter": 1000, "C": 1.0}, {"purpose": "baseline", "model_family": "linear"}),
    ("lr_regularized", LogisticRegression(max_iter=1000, C=0.1, random_state=42),
     {"model_type": "LogisticRegression", "max_iter": 1000, "C": 0.1}, {"purpose": "experiment", "model_family": "linear"}),
]

for name, model, params, tags in run_configs:
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.set_tags(tags)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mlflow.log_metrics({
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, average='weighted')
        })

print(f"✅ Created {len(run_configs)} runs!")

## Step 2: Basic Search — Get All Runs

In [ ]:
# Get all runs from our experiment
all_runs = mlflow.search_runs(experiment_names=["03_search_and_filter"])

print(f"Total runs: {len(all_runs)}")
print(f"\nColumns available: {list(all_runs.columns)[:10]}...")

# Show key columns
display_cols = ["tags.mlflow.runName", "params.model_type", 
                "metrics.accuracy", "metrics.f1", "tags.purpose"]
display_cols = [c for c in display_cols if c in all_runs.columns]
all_runs[display_cols]

## Step 3: Filter by Metrics

In [ ]:
# Find runs with accuracy > 0.95
high_acc = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    filter_string="metrics.accuracy > 0.95"
)

print(f"Runs with accuracy > 0.95: {len(high_acc)}")
if len(high_acc) > 0:
    print(high_acc[[c for c in display_cols if c in high_acc.columns]].to_string(index=False))

In [ ]:
# Find runs within a metric range
mid_range = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    filter_string="metrics.accuracy >= 0.90 AND metrics.accuracy <= 0.98"
)

print(f"Runs with 0.90 <= accuracy <= 0.98: {len(mid_range)}")
if len(mid_range) > 0:
    mid_cols = [c for c in display_cols if c in mid_range.columns]
    print(mid_range[mid_cols].to_string(index=False))

## Step 4: Filter by Parameters

In [ ]:
# Find all RandomForest runs
rf_runs = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    filter_string="params.model_type = 'RandomForest'"
)

print(f"RandomForest runs: {len(rf_runs)}")
rf_cols = [c for c in ["tags.mlflow.runName", "params.n_estimators", "params.max_depth", 
            "metrics.accuracy"] if c in rf_runs.columns]
if len(rf_runs) > 0:
    print(rf_runs[rf_cols].to_string(index=False))

In [ ]:
# Pattern matching with LIKE
tree_models = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    filter_string="params.model_type LIKE '%Tree%'"
)

print(f"Models containing 'Tree': {len(tree_models)}")
if len(tree_models) > 0:
    tree_cols = [c for c in display_cols if c in tree_models.columns]
    print(tree_models[tree_cols].to_string(index=False))

## Step 5: Filter by Tags

In [ ]:
# Find baseline runs
baselines = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    filter_string="tags.purpose = 'baseline'"
)

print(f"Baseline runs: {len(baselines)}")
if len(baselines) > 0:
    base_cols = [c for c in display_cols if c in baselines.columns]
    print(baselines[base_cols].to_string(index=False))

In [ ]:
# Combine: ensemble models that are experiments (not baselines)
ensemble_experiments = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    filter_string="tags.model_family = 'ensemble' AND tags.purpose = 'experiment'"
)

print(f"Ensemble experiment runs: {len(ensemble_experiments)}")
if len(ensemble_experiments) > 0:
    ens_cols = [c for c in display_cols if c in ensemble_experiments.columns]
    print(ensemble_experiments[ens_cols].to_string(index=False))

## Step 6: Sorting & Ordering

In [ ]:
# Get top 3 runs by accuracy
top_runs = mlflow.search_runs(
    experiment_names=["03_search_and_filter"],
    order_by=["metrics.accuracy DESC"],
    max_results=3
)

print("🏆 Top 3 runs by accuracy:")
top_cols = [c for c in display_cols if c in top_runs.columns]
print(top_runs[top_cols].to_string(index=False))

## Step 7: Building a "Best Model Finder" Function

In [ ]:
def find_best_run(experiment_name, metric="accuracy", model_type=None):
    """
    Find the best run in an experiment by a given metric.
    
    Args:
        experiment_name: Name of the MLFlow experiment
        metric: Metric to optimize (default: accuracy)
        model_type: Optional filter for model type
    
    Returns:
        Best run as a pandas Series
    """
    filter_str = ""
    if model_type:
        filter_str = f"params.model_type = '{model_type}'"
    
    runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        filter_string=filter_str if filter_str else None,
        order_by=[f"metrics.{metric} DESC"],
        max_results=1
    )
    
    if len(runs) == 0:
        print("No runs found!")
        return None
    
    best = runs.iloc[0]
    run_name = best.get("tags.mlflow.runName", "unknown")
    print(f"🏆 Best run: {run_name}")
    print(f"   {metric}: {best.get(f'metrics.{metric}', 'N/A')}")
    print(f"   Run ID: {best['run_id']}")
    return best

# Use it!
print("=== Overall Best ===")
best = find_best_run("03_search_and_filter")

print("\n=== Best RandomForest ===")
best_rf = find_best_run("03_search_and_filter", model_type="RandomForest")

print("\n=== Best by F1 ===")
best_f1 = find_best_run("03_search_and_filter", metric="f1")

## 🔑 Key Takeaways

1. **`search_runs()`** returns a pandas DataFrame — easy to analyze!
2. **Filter syntax**: `metrics.X > N`, `params.X = 'value'`, `tags.X = 'value'`
3. **Combine filters** with `AND` (no `OR` support in the filter string)
4. **`LIKE`** for pattern matching: `params.model LIKE '%Forest%'`
5. **`order_by`** for sorting: `["metrics.accuracy DESC"]`
6. **`max_results`** to limit output

---

## ➡️ Next: `04_mlflow_ui_guide.md` — Visual guide to the MLFlow UI